<a href="https://colab.research.google.com/github/photominion777/exposure-value-to-light-value/blob/main/Light_Meter_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
#@title 🎛️ Interactive Digital Light Meter Simulator (Δ LV) { display-mode: "form" }

import math
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Create an isolated output area to prevent function signature leaking
output_area = widgets.Output()

def calculate_and_display(L_input, N, t_str, S):
    # Parse shutter speed fraction string into float
    if "/" in str(t_str):
        num, denom = map(float, t_str.split("/"))
        t = num / denom
    else:
        t = float(t_str)

    K = 12.5
    L = float(L_input)

    # SYSTEM SEPARATION: Left side (Camera) vs Right side (Environment)
    av = math.log2(N**2)
    tv = math.log2(t)
    sv = math.log2(S / 100)
    LV_cam = av - tv - sv

    # Right side uses the physical luminance L
    LV_ext = math.log2(L * (100 / K))
    delta_lv = LV_ext - LV_cam

    # EXACT COURIER ALIGNMENT MATRIX
    scale_header = " -5       -4       -3       -2       -1        0       +1       +2       +3       +4       +5  "

    scale_ticks = []
    for index in range(-15, 16):
        tick_value = index / 3.0

        if abs(delta_lv - tick_value) < (1.0 / 6.0) and -5.1 < delta_lv < 5.1:
            scale_ticks.append("▲")
        elif index % 3 == 0:
            scale_ticks.append("|")
        else:
            scale_ticks.append("·")

    prefix = "◀ " if delta_lv < -5.1 else "  "
    suffix = " ▶" if delta_lv > 5.1 else "  "

    scale_visual = prefix + "  ".join(scale_ticks) + suffix

    output_text = f"""
===================================================================================================
[ INTERNAL DIGITAL LIGHT METER ANALYSIS ]
===================================================================================================
Calculated Physical Luminance (L)     : {L:.2f} cd/m²
Camera System Matrix Value (LV_cam)   : {LV_cam:.2f} stops (av: {av:.2f})
External Environmental Value (LV_ext) : {LV_ext:.2f} stops
---------------------------------------------------------------------------------------------------
Δ LV Deviation (Viewfinder Indicator) : {delta_lv:+.2f} stops
---------------------------------------------------------------------------------------------------
{scale_header}
{scale_visual}
===================================================================================================
"""

    if abs(delta_lv) < 0.17:
        output_text += "✅ EXPOSURE BALANCE: Perfectly matched. Light values are optimal."
    elif delta_lv > 0:
        output_text += "⚠️ OVEREXPOSURE: Camera configuration expects a brighter scene."
    else:
        output_text += "⚠️ UNDEREXPOSURE: Camera configuration expects a darker scene."

    # Send output exclusively to our clean container area
    with output_area:
        clear_output(wait=True)
        display(HTML(f"<pre style='font-family: Courier New, Courier, monospace; line-height: 1.2; font-size: 14px;'>{output_text}</pre>"))

# MATHEMATICAL TRICK: Generate logarithmic third-stop options for physical Luminance
K_const = 12.5
luminance_options = []
for index in range(-6, 49):
    lv_val = index / 3.0
    l_nor = 2**lv_val
    l_phys = l_nor / (100 / K_const)
    if l_phys >= 10:
        l_phys = round(l_phys, 1)
    else:
        l_phys = round(l_phys, 2)
    if l_phys not in luminance_options:
        luminance_options.append(l_phys)

# Define control elements explicitly
l_slider = widgets.SelectionSlider(options=luminance_options, value=128.0, description="Luminance (L):")
n_slider = widgets.SelectionSlider(options=[0.7, 0.95, 1.0, 1.2, 1.4, 2.0, 2.8, 4.0, 5.6, 8.0, 11.0, 16.0, 22.0], value=2.8, description="Aperture (N):")
t_slider = widgets.SelectionSlider(options=["1", "1/2", "1/4", "1/8", "1/15", "1/30", "1/60", "1/125", "1/250", "1/500", "1/1000", "1/2000", "1/4000", "1/8000"], value="1/125", description="Shutter (t):")
s_slider = widgets.IntSlider(value=400, min=100, max=12800, step=50, description="ISO Speed (S):", continuous_update=True)

# Link control updates directly to calculations
ui_controls = widgets.interactive_output(calculate_and_display, {'L_input': l_slider, 'N': n_slider, 't_str': t_slider, 'S': s_slider})

# Render interface components strictly formatted (Sliders first, then the Light Meter)
display(widgets.VBox([l_slider, n_slider, t_slider, s_slider]), output_area)


Output()